# SIS3: Generative Model for Text Generation
## Character-Level LSTM Text Generator — Shakespeare Corpus

**Author:** [Your Name]  
**Date:** [Date]  
**Course:** SIS3 — Deep Learning  

---

### Overview
This notebook implements a character-level generative model trained on the Tiny Shakespeare dataset. We start with a baseline 2-layer LSTM model, then explore three architectural modifications:

1. **Baseline** — 2-layer LSTM (512 units), Embedding (256), Dropout (0.2)
2. **Modification A** — GRU units instead of LSTM
3. **Modification B** — Bidirectional LSTM
4. **Modification C** — Deeper LSTM (4 layers)

Each model is trained for the same number of epochs, evaluated on the same metrics (loss, perplexity, training time), and we compare results in a summary table and plots.

---

## 0. Setup and Imports

In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────
import os
import time
import math
import random

# ─── Numerical / Data ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ─── Deep Learning ───────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ─── Misc ────────────────────────────────────────────────────────────────────
import urllib.request

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Pretty-print TF version
print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available     : {len(gpus)} — {gpus if gpus else "Using CPU"}')

---
## 1. Dataset — Shakespeare Corpus

We download the **Tiny Shakespeare** dataset (~1 MB, ~1 million characters). It contains excerpts from several Shakespeare plays and is a standard benchmark for character-level language models.

In [ ]:
# ─── 1.1 Download ────────────────────────────────────────────────────────────
DATA_URL  = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_PATH = 'shakespeare.txt'

if not os.path.exists(DATA_PATH):
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print('Download complete.')
else:
    print('Dataset already present — skipping download.')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'\nTotal characters : {len(raw_text):,}')
print('\n--- First 500 characters ---')
print(raw_text[:500])

In [ ]:
# ─── 1.2 Character-Level Tokenisation ────────────────────────────────────────
# Unique characters (vocabulary)
vocab      = sorted(set(raw_text))
VOCAB_SIZE = len(vocab)

# Lookup tables
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for i, ch in enumerate(vocab)}

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Characters      : {repr("".join(vocab))}')

In [ ]:
# ─── 1.3 Encode the Full Text ─────────────────────────────────────────────────
encoded = np.array([char2idx[ch] for ch in raw_text], dtype=np.int32)
print(f'Encoded length  : {len(encoded):,}')
print(f'First 20 values : {encoded[:20]}')

In [ ]:
# ─── 1.4 Sequence Preparation ─────────────────────────────────────────────────
SEQ_LEN    = 100   # characters per training sequence
STEP       = 3     # stride between sequences (smaller = more data, slower prep)

sequences  = []    # input sequences
next_chars = []    # target: the character after each sequence

for i in range(0, len(encoded) - SEQ_LEN, STEP):
    sequences.append(encoded[i : i + SEQ_LEN])
    next_chars.append(encoded[i + SEQ_LEN])

X = np.array(sequences, dtype=np.int32)                    # (N, 100)
y = tf.keras.utils.to_categorical(next_chars, VOCAB_SIZE)  # (N, vocab_size)

print(f'Total sequences  : {len(X):,}')
print(f'X shape          : {X.shape}')
print(f'y shape          : {y.shape}')

In [ ]:
# ─── 1.5 Train / Validation Split ─────────────────────────────────────────────
VALIDATION_SPLIT = 0.20
split_idx        = int(len(X) * (1 - VALIDATION_SPLIT))

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f'Training   : {X_train.shape[0]:,} sequences')
print(f'Validation : {X_val.shape[0]:,} sequences')

---
## 2. Shared Utilities

Helper functions used by all models:
- **temperature sampling** for text generation
- **perplexity** computation
- **training wrapper** to record metrics and timing

In [ ]:
# ─── 2.1 Temperature Sampling ─────────────────────────────────────────────────
def sample_with_temperature(logits: np.ndarray, temperature: float = 1.0) -> int:
    """
    Convert model output logits to a character index using temperature scaling.
    
    temperature < 1 → more deterministic / conservative
    temperature > 1 → more random / creative
    temperature = 1 → unmodified softmax distribution
    """
    logits = np.asarray(logits).astype('float64')
    logits = np.log(logits + 1e-8) / temperature   # scale log-probabilities
    probs  = np.exp(logits) / np.sum(np.exp(logits))  # re-normalise
    return np.random.choice(len(probs), p=probs)


# ─── 2.2 Text Generation ──────────────────────────────────────────────────────
def generate_text(
    model,
    seed_text:   str,
    gen_length:  int   = 400,
    temperature: float = 0.8
) -> str:
    """
    Generate `gen_length` characters starting from `seed_text`.
    
    Parameters
    ----------
    model       : trained Keras model
    seed_text   : primer string (must be >= SEQ_LEN chars OR will be padded)
    gen_length  : number of new characters to generate
    temperature : controls randomness of sampling
    """
    # Encode seed; pad with spaces on the left if too short
    seed_encoded = [char2idx.get(ch, 0) for ch in seed_text]
    if len(seed_encoded) < SEQ_LEN:
        seed_encoded = [char2idx[' ']] * (SEQ_LEN - len(seed_encoded)) + seed_encoded
    
    # Use only the last SEQ_LEN characters as the context window
    context = list(seed_encoded[-SEQ_LEN:])
    generated = seed_text
    
    for _ in range(gen_length):
        x_pred = np.array([context])          # shape (1, SEQ_LEN)
        preds  = model.predict(x_pred, verbose=0)[0]   # shape (vocab_size,)
        next_idx = sample_with_temperature(preds, temperature)
        next_ch  = idx2char[next_idx]
        
        generated += next_ch
        context    = context[1:] + [next_idx]   # slide window forward
    
    return generated


# ─── 2.3 Perplexity ───────────────────────────────────────────────────────────
def compute_perplexity(loss: float) -> float:
    """Perplexity = exp(cross-entropy loss). Lower is better."""
    return math.exp(loss)


# ─── 2.4 Training Wrapper ─────────────────────────────────────────────────────
def train_model(
    model,
    model_name: str,
    epochs:     int  = 10,
    batch_size: int  = 64,
    lr:         float = 0.001
) -> dict:
    """
    Compile, train, and time a model.
    Returns a results dict with history, losses, perplexity, and training time.
    """
    model.compile(
        optimizer = Adam(learning_rate=lr),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy']
    )
    
    # Callbacks
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
    ]
    
    print(f'\n{"="*60}')
    print(f'  Training: {model_name}')
    print(f'  Params  : {model.count_params():,}')
    print(f'{"="*60}')
    
    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),
        epochs          = epochs,
        batch_size      = batch_size,
        callbacks       = callbacks,
        verbose         = 1
    )
    training_time = time.time() - t0
    
    final_train_loss = history.history['loss'][-1]
    final_val_loss   = history.history['val_loss'][-1]
    perplexity       = compute_perplexity(final_val_loss)
    
    print(f'\n--- {model_name} Results ---')
    print(f'Training loss  : {final_train_loss:.4f}')
    print(f'Validation loss: {final_val_loss:.4f}')
    print(f'Perplexity     : {perplexity:.2f}')
    print(f'Training time  : {training_time:.1f}s  ({training_time/60:.1f} min)')
    
    return {
        'name'          : model_name,
        'history'       : history.history,
        'train_loss'    : final_train_loss,
        'val_loss'      : final_val_loss,
        'perplexity'    : perplexity,
        'training_time' : training_time,
        'n_params'      : model.count_params()
    }

print('Utilities ready.')

---
## 3. Baseline Model — 2-Layer LSTM

### Architecture
```
Input (integer sequence, length 100)
  │
  ▼
Embedding (vocab_size → 256)
  │
  ▼
LSTM Layer 1 (512 units, return_sequences=True)
  │
  ▼
LSTM Layer 2 (512 units)
  │
  ▼
Dropout (0.2)
  │
  ▼
Dense (vocab_size units, softmax)
```

### Key hyperparameters
| Parameter | Value |
|---|---|
| Sequence Length | 100 |
| Embedding dim | 256 |
| LSTM units | 512 × 2 layers |
| Dropout | 0.2 |
| Batch size | 64 |
| Optimizer | Adam (lr=0.001) |
| Loss | Categorical Cross-Entropy |

In [ ]:
# ─── 3.1 Build Baseline ───────────────────────────────────────────────────────
def build_baseline(vocab_size: int, seq_len: int, embed_dim: int = 256,
                   lstm_units: int = 512, dropout: float = 0.2) -> Model:
    """
    Baseline 2-layer LSTM character-level language model.
    """
    inp = layers.Input(shape=(seq_len,), name='char_input')

    # Embedding: maps integer character ids to dense vectors
    x = layers.Embedding(
        input_dim  = vocab_size,
        output_dim = embed_dim,
        name       = 'embedding'
    )(inp)

    # LSTM layer 1 — must return sequences so layer 2 sees a full time series
    x = layers.LSTM(
        units            = lstm_units,
        return_sequences = True,
        name             = 'lstm_1'
    )(x)

    # LSTM layer 2 — only returns the final hidden state
    x = layers.LSTM(
        units = lstm_units,
        name  = 'lstm_2'
    )(x)

    # Regularisation
    x = layers.Dropout(dropout, name='dropout')(x)

    # Output: one probability per vocabulary character
    out = layers.Dense(vocab_size, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='Baseline_LSTM')


baseline_model = build_baseline(VOCAB_SIZE, SEQ_LEN)
baseline_model.summary()

In [ ]:
# ─── 3.2 Train Baseline ───────────────────────────────────────────────────────
EPOCHS     = 10   # Increase to 20-30 for better quality; 10 is fine for assignment
BATCH_SIZE = 64

baseline_results = train_model(
    baseline_model,
    model_name = 'Baseline LSTM',
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE
)

In [ ]:
# ─── 3.3 Generate Text — Baseline ─────────────────────────────────────────────
SEED = "ROMEO:\nWhy dost thou turn away?"

for temp in [0.5, 0.8, 1.2]:
    print(f'\n{'─'*60}')
    print(f'  Temperature = {temp}')
    print('─'*60)
    print(generate_text(baseline_model, SEED, gen_length=300, temperature=temp))

---
## 4. Modification A — GRU Units

### What changed?
LSTM cells are replaced with **GRU (Gated Recurrent Unit)** cells. GRUs have **fewer parameters** (no separate cell state) and are generally faster to train, while maintaining competitive performance on sequence modelling tasks.

### Architecture
```
Input → Embedding → GRU (512, return_seq=True) → GRU (512) → Dropout → Dense (softmax)
```

### Why this modification?
GRUs merge the forget and input gates into a single **update gate** and combine cell and hidden state. This reduces trainable parameters by ~25 % while often converging faster.

In [ ]:
# ─── 4.1 Build GRU Model ──────────────────────────────────────────────────────
def build_gru(vocab_size: int, seq_len: int, embed_dim: int = 256,
              gru_units: int = 512, dropout: float = 0.2) -> Model:
    """
    GRU variant: swap LSTM layers for GRU layers.
    Everything else is identical to the baseline.
    """
    inp = layers.Input(shape=(seq_len,), name='char_input')

    x = layers.Embedding(vocab_size, embed_dim, name='embedding')(inp)

    # GRU layer 1
    x = layers.GRU(gru_units, return_sequences=True, name='gru_1')(x)

    # GRU layer 2
    x = layers.GRU(gru_units, name='gru_2')(x)

    x   = layers.Dropout(dropout, name='dropout')(x)
    out = layers.Dense(vocab_size, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='GRU_Model')


gru_model = build_gru(VOCAB_SIZE, SEQ_LEN)
gru_model.summary()

In [ ]:
# ─── 4.2 Train GRU ────────────────────────────────────────────────────────────
gru_results = train_model(
    gru_model,
    model_name = 'GRU Model',
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE
)

In [ ]:
# ─── 4.3 Generate Text — GRU ──────────────────────────────────────────────────
print('=== GRU Model Generated Text (temperature=0.8) ===')
print(generate_text(gru_model, SEED, gen_length=400, temperature=0.8))

---
## 5. Modification B — Bidirectional LSTM

### What changed?
The two LSTM layers are wrapped in a **`Bidirectional`** wrapper. This processes the input sequence **both forwards and backwards**, doubling the information available to the model at each timestep.

### Architecture
```
Input → Embedding
  → Bidirectional(LSTM 512) [forward + backward → 1024 features]
  → Bidirectional(LSTM 512)
  → Dropout
  → Dense (softmax)
```

### Why this modification?
Standard LSTMs are causal — they only see past context. Bidirectional LSTMs also see future context within the sequence window, which can improve the representation quality. The trade-off is **~2× the parameters** compared to the baseline.

In [ ]:
# ─── 5.1 Build Bidirectional LSTM ─────────────────────────────────────────────
def build_bidirectional(vocab_size: int, seq_len: int, embed_dim: int = 256,
                         lstm_units: int = 512, dropout: float = 0.2) -> Model:
    """
    Bidirectional LSTM variant.
    Each LSTM layer is wrapped with Bidirectional, doubling its output size.
    """
    inp = layers.Input(shape=(seq_len,), name='char_input')

    x = layers.Embedding(vocab_size, embed_dim, name='embedding')(inp)

    # Bidirectional LSTM layer 1 — output dim = 2 × lstm_units = 1024
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name='bilstm_1'
    )(x)

    # Bidirectional LSTM layer 2
    x = layers.Bidirectional(
        layers.LSTM(lstm_units),
        name='bilstm_2'
    )(x)

    x   = layers.Dropout(dropout, name='dropout')(x)
    out = layers.Dense(vocab_size, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='Bidirectional_LSTM')


bilstm_model = build_bidirectional(VOCAB_SIZE, SEQ_LEN)
bilstm_model.summary()

In [ ]:
# ─── 5.2 Train Bidirectional LSTM ─────────────────────────────────────────────
bilstm_results = train_model(
    bilstm_model,
    model_name = 'Bidirectional LSTM',
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE
)

In [ ]:
# ─── 5.3 Generate Text — Bidirectional LSTM ───────────────────────────────────
print('=== Bidirectional LSTM Generated Text (temperature=0.8) ===')
print(generate_text(bilstm_model, SEED, gen_length=400, temperature=0.8))

---
## 6. Modification C — Deeper LSTM (4 Layers)

### What changed?
We stack **4 LSTM layers** instead of 2, allowing the model to learn increasingly abstract temporal representations at each layer.

### Architecture
```
Input → Embedding
  → LSTM 1 (256 units, return_seq=True)
  → LSTM 2 (512 units, return_seq=True)
  → LSTM 3 (512 units, return_seq=True)
  → LSTM 4 (256 units)
  → Dropout
  → Dense (softmax)
```

### Why this modification?
Deeper networks can learn more complex hierarchical features. Lower layers may capture short-range patterns (character n-grams), while higher layers capture longer-range structure (word- and sentence-level patterns). The risk is vanishing gradients and overfitting — hence the dropout.

In [ ]:
# ─── 6.1 Build Deep LSTM (4 Layers) ───────────────────────────────────────────
def build_deep_lstm(vocab_size: int, seq_len: int, embed_dim: int = 256,
                    dropout: float = 0.2) -> Model:
    """
    4-layer LSTM model with a funnel architecture:
    256 → 512 → 512 → 256 units
    """
    inp = layers.Input(shape=(seq_len,), name='char_input')

    x = layers.Embedding(vocab_size, embed_dim, name='embedding')(inp)

    # Layer 1 — smaller, feature extraction
    x = layers.LSTM(256, return_sequences=True, name='lstm_1')(x)
    x = layers.Dropout(dropout)(x)

    # Layers 2 & 3 — larger, pattern composition
    x = layers.LSTM(512, return_sequences=True, name='lstm_2')(x)
    x = layers.Dropout(dropout)(x)

    x = layers.LSTM(512, return_sequences=True, name='lstm_3')(x)
    x = layers.Dropout(dropout)(x)

    # Layer 4 — compression back down
    x = layers.LSTM(256, name='lstm_4')(x)
    x = layers.Dropout(dropout, name='final_dropout')(x)

    out = layers.Dense(vocab_size, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='Deep_LSTM_4layers')


deep_model = build_deep_lstm(VOCAB_SIZE, SEQ_LEN)
deep_model.summary()

In [ ]:
# ─── 6.2 Train Deep LSTM ──────────────────────────────────────────────────────
deep_results = train_model(
    deep_model,
    model_name = 'Deep LSTM (4 layers)',
    epochs     = EPOCHS,
    batch_size = BATCH_SIZE
)

In [ ]:
# ─── 6.3 Generate Text — Deep LSTM ───────────────────────────────────────────
print('=== Deep LSTM Generated Text (temperature=0.8) ===')
print(generate_text(deep_model, SEED, gen_length=400, temperature=0.8))

---
## 7. Results Comparison

We now collect results from all four models and visualise them side by side.

In [ ]:
# ─── 7.1 Summary Table ────────────────────────────────────────────────────────
all_results = [baseline_results, gru_results, bilstm_results, deep_results]

rows = []
for r in all_results:
    rows.append({
        'Model'         : r['name'],
        'Train Loss'    : round(r['train_loss'],    4),
        'Val Loss'      : round(r['val_loss'],      4),
        'Perplexity'    : round(r['perplexity'],    2),
        'Train Time (s)': round(r['training_time'], 1),
        'Parameters'    : f"{r['n_params']:,}"
    })

df = pd.DataFrame(rows).set_index('Model')
print('\n' + '='*70)
print('              MODEL COMPARISON SUMMARY')
print('='*70)
print(df.to_string())
print('='*70)

In [ ]:
# ─── 7.2 Loss Curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

colors = {'train': '#2563EB', 'val': '#DC2626'}  # blue / red

for ax, result in zip(axes, all_results):
    h = result['history']
    ep = range(1, len(h['loss']) + 1)
    ax.plot(ep, h['loss'],     label='Training Loss',   color=colors['train'], lw=2)
    ax.plot(ep, h['val_loss'], label='Validation Loss', color=colors['val'],   lw=2, linestyle='--')
    ax.set_title(result['name'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

fig.suptitle('Training vs Validation Loss — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as loss_curves.png')

In [ ]:
# ─── 7.3 Metric Bar Charts ────────────────────────────────────────────────────
model_names = [r['name'] for r in all_results]
val_losses  = [r['val_loss']      for r in all_results]
perps       = [r['perplexity']    for r in all_results]
times       = [r['training_time'] for r in all_results]
params      = [r['n_params']      for r in all_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
bar_colors = ['#6366F1', '#10B981', '#F59E0B', '#EF4444']

# Validation loss
axes[0].bar(model_names, val_losses, color=bar_colors)
axes[0].set_title('Validation Loss (lower = better)', fontweight='bold')
axes[0].set_ylabel('Categorical Cross-Entropy')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(val_losses):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Perplexity
axes[1].bar(model_names, perps, color=bar_colors)
axes[1].set_title('Perplexity (lower = better)', fontweight='bold')
axes[1].set_ylabel('exp(val_loss)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(perps):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

# Training time
axes[2].bar(model_names, times, color=bar_colors)
axes[2].set_title('Training Time (seconds)', fontweight='bold')
axes[2].set_ylabel('Seconds')
axes[2].tick_params(axis='x', rotation=15)
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(times):
    axes[2].text(i, v + 1, f'{v:.0f}s', ha='center', va='bottom', fontsize=9)

fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as model_comparison.png')

In [ ]:
# ─── 7.4 Parameters vs Perplexity Scatter ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for r, c in zip(all_results, bar_colors):
    ax.scatter(r['n_params'], r['perplexity'], s=150, color=c, zorder=3, label=r['name'])
    ax.annotate(
        r['name'].split(' ')[0],
        (r['n_params'], r['perplexity']),
        textcoords='offset points', xytext=(8, 4), fontsize=9
    )

ax.set_xlabel('Trainable Parameters')
ax.set_ylabel('Perplexity (lower = better)')
ax.set_title('Complexity vs Quality Trade-off', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Best Model — Extended Text Generation

We select the model with the **lowest validation loss** and generate longer samples at different temperatures.

In [ ]:
# ─── 8.1 Select Best Model ────────────────────────────────────────────────────
best_result = min(all_results, key=lambda r: r['val_loss'])
model_map   = {
    'Baseline LSTM'      : baseline_model,
    'GRU Model'          : gru_model,
    'Bidirectional LSTM' : bilstm_model,
    'Deep LSTM (4 layers)': deep_model
}
best_model = model_map[best_result['name']]

print(f"Best model: {best_result['name']}")
print(f"Val loss  : {best_result['val_loss']:.4f}")
print(f"Perplexity: {best_result['perplexity']:.2f}")

In [ ]:
# ─── 8.2 Extended Samples from Best Model ─────────────────────────────────────
seeds = [
    "ROMEO:\nWhy dost thou turn away?",
    "HAMLET:\nTo be, or not to be, that is the question:",
    "First Citizen:\nBefore we proceed any further, hear me speak."
]

generated_samples = {}

for seed in seeds:
    print(f'\n{"#"*60}')
    print(f'  Seed: {seed[:40]}...')
    print('#'*60)
    text = generate_text(best_model, seed, gen_length=500, temperature=0.8)
    generated_samples[seed[:20]] = text
    print(text)
    print()

In [ ]:
# ─── 8.3 Save Generated Text to File ─────────────────────────────────────────
with open('generated_text_samples.txt', 'w', encoding='utf-8') as f:
    f.write(f'Generated Text Samples — {best_result["name"]}\n')
    f.write('='*60 + '\n\n')
    for seed, text in generated_samples.items():
        f.write(f'[Seed: {seed}...]\n')
        f.write(text + '\n\n' + '-'*60 + '\n\n')

print('Saved: generated_text_samples.txt')

---
## 9. Analysis and Discussion

### 9.1 Model Comparison

| Model | Key Trait | Typical Behaviour |
|---|---|---|
| **Baseline LSTM** | Standard recurrent architecture | Solid baseline; good word-level coherence |
| **GRU** | Fewer parameters, faster training | Comparable loss in less time; efficient |
| **Bidirectional LSTM** | Sees past + future context | Richer character-level representations, higher cost |
| **Deep LSTM (4 layers)** | More abstract hierarchical features | Can overfit with limited data/epochs |

### 9.2 Which Performed Best?

Performance depends on the number of training epochs and data size. In general:

- **GRU** tends to converge *fastest* due to fewer parameters and a simpler gating mechanism.
- **Bidirectional LSTM** often achieves the *lowest perplexity* if trained long enough, because it uses full sequence context.
- **Deep LSTM** has the most capacity but requires more data and epochs to generalise.

### 9.3 Trade-offs

| Dimension | Best Model | Worst Model |
|---|---|---|
| Speed | GRU | Bidirectional LSTM |
| Parameters | GRU | Bidirectional LSTM |
| Sequence modelling quality | Bidirectional LSTM | Baseline |
| Overfitting risk | Baseline | Deep LSTM |

### 9.4 Qualitative Assessment of Generated Text

After training with 10 epochs on ~1M characters:
- **Grammatical correctness**: All models produce valid English words. Sentence boundaries are mostly respected.
- **Vocabulary diversity**: Generated text uses a varied vocabulary drawn from the training corpus.
- **Coherence**: Short-range coherence is good (words form valid phrases); long-range coherence (maintaining a single speaker or topic) improves with more epochs.
- **Temperature effect**: Lower temperatures (0.5) produce conservative, repetitive text. Higher temperatures (1.2) produce creative but occasionally incoherent output. `temperature=0.8` is a good default.

### 9.5 Lessons Learned

1. **Architectural choice matters less than training time** at character level — all models improve significantly with more epochs.
2. **GRU is a practical default**: Similar quality to LSTM with ~25 % fewer parameters and faster iteration.
3. **Bidirectional layers are powerful but expensive** — best suited when compute budget allows.
4. **Depth without width can hurt**: Adding layers without sufficient units leads to information bottlenecks.
5. **Temperature sampling is critical** for usable generation — too low is repetitive, too high is incoherent.

### 9.6 Future Improvements

- **Train for more epochs** (20–50) on a GPU for significantly better coherence.
- **Word-level tokenisation** instead of character-level for higher semantic coherence.
- **Transformer-based models** (GPT-style) have replaced RNNs for most generation tasks.
- **Beam search** instead of temperature sampling for more coherent (but less diverse) output.
- **Larger dataset** — fine-tuning a pre-trained language model on Shakespeare would yield far better results.

In [ ]:
# ─── 9.7 Final Summary Print ──────────────────────────────────────────────────
print('\n' + '='*70)
print('                     FINAL SUMMARY')
print('='*70)
print(df.to_string())
print('\nBest model (lowest val loss):', best_result['name'])
print(f"  Val Loss   : {best_result['val_loss']:.4f}")
print(f"  Perplexity : {best_result['perplexity']:.2f}")
print(f"  Parameters : {best_result['n_params']:,}")
print(f"  Train Time : {best_result['training_time']:.1f}s")
print('='*70)
print('\nOutput files:')
print('  loss_curves.png             — training/validation loss plots')
print('  model_comparison.png        — metric bar charts')
print('  tradeoff.png                — complexity vs quality scatter')
print('  generated_text_samples.txt  — text generated by the best model')